<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Electromagnetic_Spectrum_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Electromagnetic Spectrum Intelligence Platform
This notebook implements a comprehensive suite for RF signal simulation, radar modeling, and electronic warfare analysis using deep learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import torch
import torch.nn as nn

def generate_iq_signal(fs, duration, freq, modulation='cw', noise_snr_db=None):
    t = np.arange(0, duration, 1/fs)
    if modulation == 'cw':
        iq = np.exp(1j * 2 * np.pi * freq * t)
    elif modulation == 'lfm':
        iq = signal.chirp(t, f0=freq, f1=freq*2, t1=duration, method='linear', phi=-90)
        iq = np.exp(1j * iq)

    if noise_snr_db is not None:
        sig_avg_watts = np.mean(np.abs(iq)**2)
        sig_avg_db = 10 * np.log10(sig_avg_watts)
        noise_avg_db = sig_avg_db - noise_snr_db
        noise_avg_watts = 10 ** (noise_avg_db / 10)
        noise = np.sqrt(noise_avg_watts/2) * (np.random.randn(len(t)) + 1j*np.random.randn(len(t)))
        iq += noise
    return t, iq

def plot_spectrogram(iq, fs, title='Spectrogram'):
    f, t, Sxx = signal.spectrogram(iq, fs, return_onesided=False)
    plt.figure(figsize=(10, 6))
    plt.pcolormesh(t, np.fft.fftshift(f), np.fft.fftshift(10 * np.log10(Sxx), axes=0), shading='gouraud')
    plt.ylabel('Frequency [Hz]')
    plt.xlabel('Time [sec]')
    plt.title(title)
    plt.colorbar(label='Intensity [dB]')
    plt.show()

In [ ]:
# Demonstration of signal generation and visualization
fs = 1e6  # 1MHz sampling rate
duration = 0.01
t, iq_sig = generate_iq_signal(fs, duration, freq=1e5, modulation='lfm', noise_snr_db=20)
plot_spectrogram(iq_sig, fs, title='LFM Signal Spectrogram (Waterfall)')

## Signal Classification with CNNs
In this section, we build a CNN to classify modulation types. We convert complex IQ samples into a two-channel input (Real and Imaginary) for the network.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class ModulationCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(ModulationCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 250, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

def create_dataset(num_samples=100):
    data, labels = [], []
    for _ in range(num_samples // 2):
        _, iq_cw = generate_iq_signal(1e6, 0.001, 1e5, 'cw', noise_snr_db=10)
        data.append(np.stack([iq_cw.real, iq_cw.imag]))
        labels.append(0)
        _, iq_lfm = generate_iq_signal(1e6, 0.001, 1e5, 'lfm', noise_snr_db=10)
        data.append(np.stack([iq_lfm.real, iq_lfm.imag]))
        labels.append(1)
    return torch.tensor(np.array(data), dtype=torch.float32), torch.tensor(np.array(labels))

model = ModulationCNN(num_classes=2)
X, y = create_dataset()
print(f"Dataset created: {X.shape}")

Dataset created: torch.Size([100, 2, 1001])


## Radar Simulation and Electronic Warfare (EW)
This section models pulse-doppler radar signals and implements jamming detection using Autoencoders.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class JammingAutoencoder(nn.Module):
    def __init__(self, input_dim=1000):
        super(JammingAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

def simulate_radar_pulse(pri, pulse_width, fs, num_pulses):
    t_pulse = np.arange(0, pulse_width, 1/fs)
    pulse = np.exp(1j * np.pi * (1e6/pulse_width) * t_pulse**2)
    full_signal = []
    samples_pri = int(pri * fs)
    for _ in range(num_pulses):
        padded_pulse = np.zeros(samples_pri, dtype=complex)
        padded_pulse[:len(pulse)] = pulse
        full_signal.append(padded_pulse)
    return np.concatenate(full_signal)

radar_sig = simulate_radar_pulse(pri=1e-4, pulse_width=1e-5, fs=1e7, num_pulses=5)
print(f"Radar Simulation Ready: {len(radar_sig)} samples")

Radar Simulation Ready: 5000 samples


## Advanced AI Models: Transformers, LSTMs, and Diffusion
This section implements Transformer and LSTM architectures for signal classification, and a basic Diffusion model structure for signal synthesis.

In [ ]:
class SignalTransformer(nn.Module):
    def __init__(self, input_dim=2, model_dim=64, num_heads=4, num_layers=2):
        super(SignalTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, model_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(model_dim, 2)

    def forward(self, x):
        # x shape: (batch, channels, seq_len) -> (batch, seq_len, model_dim)
        x = x.transpose(1, 2)
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x.mean(dim=1))

class SignalLSTM(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, num_layers=2):
        super(SignalLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

# Basic UNet-style Diffusion Backbone for 1D signals
class SimpleDiffusion(nn.Module):
    def __init__(self):
        super(SimpleDiffusion, self).__init__()
        self.net = nn.Sequential(
            nn.Conv1d(2, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 2, 3, padding=1)
        )
    def forward(self, x, t):
        return self.net(x)

print("Transformer, LSTM, and Diffusion models initialized.")

Transformer, LSTM, and Diffusion models initialized.


## Spectrum Analysis & Interactive Dashboard
This section provides utilities for high-resolution spectral analysis and a summary dashboard of the platform's intelligence capabilities.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def interactive_spectrum_analysis(fs=1e6):
    freq_slider = widgets.FloatSlider(value=1e5, min=0, max=5e5, step=1e4, description='Freq (Hz):')
    snr_slider = widgets.IntSlider(value=20, min=-10, max=40, description='SNR (dB):')
    mod_dropdown = widgets.Dropdown(options=['cw', 'lfm'], value='cw', description='Modulation:')

    def update_view(freq, snr, mod):
        _, iq = generate_iq_signal(fs, 0.01, freq, mod, noise_snr_db=snr)
        plot_spectrogram(iq, fs, title=f'Live Spectrum: {mod.upper()} @ {freq/1e3}kHz')

    out = widgets.interactive_output(update_view, {'freq': freq_slider, 'snr': snr_slider, 'mod': mod_dropdown})
    display(widgets.VBox([widgets.HBox([freq_slider, snr_slider, mod_dropdown]), out]))

# Launch Dashboard
interactive_spectrum_analysis()